# GX_04 — Predicting Who Will Leave (Modelling)

**The question:** *which customers are most likely to leave the bank next, and how well can we predict it before it happens?* GX_03 described who left; here we turn that into a **churn risk score** (0–1) for every customer, so the bank can contact the riskiest first.

**Inputs:** `gold_churn_features` (10,000 customers, with GX_02's fixed `split` column). **Outputs:** no new tables — the chosen model, its scores and the conclusions.

**The plan:**
1. **Check** the fixed train / validation / test split and reuse its shared preparation steps.
2. **Compare** five baseline models with default settings.
3. **Tune** the two best — the neural network and Gradient-Boosted Trees — with grid searches selected by cross-validated **AUC-PR**.
4. **Stack** them — on hold: code kept, skipped when the notebook runs.
5. **Choose** the final model — the MLP classifier — and the operating threshold — where to cut its probability into a contact list, on validation only.

**Three rules for reading the results:**
- The **test set is opened once, at the very end** — everything before that uses only training and validation.
- **Training scores never select a model** — selection uses validation.
- One headline metric everywhere: **validation AUC-PR**, read against the ≈ 0.204 random-ranking floor (Step 2).

## Step 0 — Preparing the notebook

**What we do:** load the shared helper code from `src/`, one colour palette for every chart, and the model classes we need.

**Why:**
- The preparation steps (text → numbers, scaling) are exactly the ones defined in `src/ml.py` in Phase 2. We import and reuse them instead of writing new ones, so the model sees the same 16 inputs GX_02 planned — nothing is re-invented, and nothing can drift between phases.
- `SEED = 42` from `src/config.py` is used for every random operation (tree building, cross-validation folds), so running the notebook again gives the same results.
- The colours match the other notebooks, so the slides all look alike.
- **One environment setting:** Spark ML's `CrossValidator` (Steps 4–5) caches intermediate DataFrames. On serverless compute that cache must live in a **Unity Catalog volume**, so the setup cell creates an empty scratch volume (`ml_tmp`) and points Spark ML at it via the `SPARKML_TEMP_DFS_PATH` environment variable. The volume holds only temporary Spark files, never project data.

In [0]:
# Make the shared helpers in src/ importable (this notebook lives in notebooks/).
import os
import sys

REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pyspark.ml import Pipeline
from pyspark.ml.classification import (
    GBTClassifier, LinearSVC, LogisticRegression,
    MultilayerPerceptronClassifier, RandomForestClassifier,
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import Window
from pyspark.sql import functions as F

from src.config import EXPECTED_ROWS, GOLD_CHURN, SEED
from src.ml import (
    CATEGORICAL_FEATURES, KEY_COL, LABEL_COL, NUMERIC_FEATURES, SPLIT_COL,
    build_preprocessing_stages,
)

# One colour palette for every chart (same as the other notebooks)
COLOR_MAIN = "#27ae60"    # general bars / validation scores
COLOR_DARK = "#0b5345"    # highlight / training scores
COLOR_GREY = "#7f8c8d"    # reference line

print("Spark version:", spark.version)

# CrossValidator (Steps 4-5) caches intermediate DataFrames. On serverless compute
# Spark ML needs a Unity Catalog volume for that cache, set via this environment
# variable. The volume holds only temporary Spark files, never project data.
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.churn.ml_tmp")
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/churn/ml_tmp"
print("Spark ML temp path:", os.environ["SPARKML_TEMP_DFS_PATH"])

## Step 1 — The data and the fixed split

**What we do:** read `gold_churn_features` and take the `train` / `val` / `test` parts GX_02 fixed — a deterministic hash of the customer id, already verified there and never re-made.

**Why a quick re-check:** GX_02 proved the split when it built it; this notebook confirms what it actually consumes — the gold table was already rebuilt once in this project, and a broken split is the one mistake that silently makes every model look better than it is. One cheap check re-confirms all 10,000 customers, each in exactly one part, and the ~20.4% churn rate everywhere — and stops before any model is fitted if anything changed.

The three parts are `cache()`d because training, cross-validation and scoring reuse them many times.

In [0]:
gold = spark.table(GOLD_CHURN).cache()
gold.createOrReplaceTempView("gold")

train = gold.filter(F.col(SPLIT_COL) == "train")
val = gold.filter(F.col(SPLIT_COL) == "val")
test = gold.filter(F.col(SPLIT_COL) == "test")

display(spark.sql(f'''
    SELECT {SPLIT_COL} AS split,
           COUNT(*)                         AS customers,
           SUM({LABEL_COL})                 AS churned,
           ROUND(AVG({LABEL_COL}) * 100, 2) AS churn_rate_pct
    FROM gold
    GROUP BY {SPLIT_COL}
    ORDER BY CASE {SPLIT_COL} WHEN 'train' THEN 1 WHEN 'val' THEN 2 ELSE 3 END
'''))

# Overlap check: the same customer must never appear in two splits.
total_rows = gold.count()
distinct_customers = gold.select(KEY_COL).distinct().count()
customers_in_two_splits = (
    gold.groupBy(KEY_COL)
    .agg(F.count_distinct(SPLIT_COL).alias("n_splits"))
    .filter(F.col("n_splits") > 1)
    .count()
)

print(f"Rows: {total_rows:,} | Distinct customers: {distinct_customers:,}")
print(f"Customers appearing in more than one split: {customers_in_two_splits}")
assert (
    total_rows == distinct_customers == EXPECTED_ROWS["churn"] and customers_in_two_splits == 0
), "Split broken - stop before modelling."

**Result:** the split is exactly the one GX_02 built.

| Split | Customers | Leavers | Churn rate |
|---|---|---|---|
| train | 7,000 | 1,426 | 20.4% |
| validation | 1,499 | 305 | 20.4% |
| test | 1,501 | 306 | 20.4% |

10,000 distinct customers, **0 in more than one split**, and the same ~20.4% churn rate everywhere — training and judging face the same imbalance, and the random-ranking AUC-PR floor is ≈ 0.204 in all three parts.

## Step 2 — How we judge the models: AUC-PR

**The headline metric for this notebook is validation AUC-PR**, with churn-class precision, recall and F1 alongside; accuracy is kept only as context.

**Why not accuracy:** only ~20.4% of customers leave, so predicting *stays* for everyone is already 79.6% accurate while catching none of the leavers. Accuracy mostly measures the majority class.

**Why AUC-PR:** precision–recall follows the **churn class only** — of the customers flagged, how many really leave (precision); of the leavers, how many get flagged (recall) — and AUC-PR summarises that trade-off across **every threshold** in one number, with no threshold choice needed. That is exactly how a churn-risk ranking should be judged: the bank contacts the riskiest first, so the top of the list must be the likeliest leavers. The floor is a model that ranks at random: **AUC-PR ≈ the churn rate ≈ 0.204** — the dashed line in every chart.

In [0]:
# Step 2's headline metric — validation AUC-PR. Also the selection metric for the grid
# searches in Steps 4-5, inside cross-validation on the training split.
auc_pr_eval = BinaryClassificationEvaluator(
    labelCol=LABEL_COL, rawPredictionCol="probability", metricName="areaUnderPR"
)
auc_roc_eval = BinaryClassificationEvaluator(
    labelCol=LABEL_COL, rawPredictionCol="probability", metricName="areaUnderROC"
)

# The do-nothing floor every AUC-PR number is read against (Step 2): a model that
# ranked customers at random scores AUC-PR ≈ the churn rate. For context, "always
# predict stayed" would be 79.6% accurate while catching no leaver at all.
CHURN_RATE = round(float(val.agg(F.avg(LABEL_COL)).first()[0]), 4)
print(f"Churn rate in validation = random-ranking AUC-PR floor: {CHURN_RATE}")

results = []   # one row per model and stage ("baseline" / "tuned"); filled throughout


def make_pipeline(classifier):
    """GX_02's preparation steps + one classifier, as one Spark ML Pipeline.

    The preparation stages are UNFITTED: the pipeline learns them from whatever rows
    .fit() is called on. Fitted on `train`, the indexer and scaler see only the 7,000
    training rows; inside cross-validation each fold fits them again on that fold's
    training rows only. Validation and test rows never teach the preprocessing anything.
    """
    return Pipeline(
        stages=build_preprocessing_stages(CATEGORICAL_FEATURES, NUMERIC_FEATURES) + [classifier]
    )


def churn_metrics(preds):
    """All metrics of a prediction DataFrame (needs label, probability, prediction).

    Precision / recall / F1 are for the CHURN class only, computed by hand from the
    confusion matrix at Spark's default threshold (0.5).
    """
    df = preds.withColumn("p_churn", vector_to_array("probability")[1].cast("double"))
    cm = df.agg(
        F.sum(((F.col(LABEL_COL) == 1) & (F.col("prediction") == 1)).cast("int")).alias("tp"),
        F.sum(((F.col(LABEL_COL) == 0) & (F.col("prediction") == 1)).cast("int")).alias("fp"),
        F.sum(((F.col(LABEL_COL) == 1) & (F.col("prediction") == 0)).cast("int")).alias("fn"),
        F.sum(((F.col(LABEL_COL) == 0) & (F.col("prediction") == 0)).cast("int")).alias("tn"),
    ).first()
    tp, fp, fn, tn = (int(cm[c]) for c in ("tp", "fp", "fn", "tn"))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {
        "auc_pr": round(auc_pr_eval.evaluate(df), 4),
        "accuracy": round((tp + tn) / (tp + fp + fn + tn), 4),
        "auc_roc": round(auc_roc_eval.evaluate(df), 4),
        "precision_churn": round(precision, 4),
        "recall_churn": round(recall, 4),
        "f1_churn": round(2 * precision * recall / (precision + recall), 4) if precision + recall else 0.0,
        "confusion": {"TP": tp, "FP": fp, "FN": fn, "TN": tn},
    }


def prediction_scores(preds):
    """Accuracy and the churn-class precision / recall / F1, computed by hand from the
    confusion matrix of the `prediction` column. Works for every model — including
    the Linear SVM, which produces no probability column at all."""
    cm = preds.agg(
        F.sum(((F.col(LABEL_COL) == 1) & (F.col("prediction") == 1)).cast("int")).alias("tp"),
        F.sum(((F.col(LABEL_COL) == 0) & (F.col("prediction") == 1)).cast("int")).alias("fp"),
        F.sum(((F.col(LABEL_COL) == 1) & (F.col("prediction") == 0)).cast("int")).alias("fn"),
        F.sum(((F.col(LABEL_COL) == 0) & (F.col("prediction") == 0)).cast("int")).alias("tn"),
    ).first()
    tp, fp, fn, tn = (int(cm[c]) for c in ("tp", "fp", "fn", "tn"))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {
        "accuracy": round((tp + tn) / (tp + fp + fn + tn), 4),
        "precision_churn": round(precision, 4),
        "recall_churn": round(recall, 4),
        "f1_churn": round(2 * precision * recall / (precision + recall), 4) if precision + recall else 0.0,
    }


def score_row(model_name, train_preds, val_preds, stage):
    """One comparison row: train and val AUC-PR and accuracy, the gap, and validation extras."""
    m_val = churn_metrics(val_preds)
    row = {
        "model": model_name,
        "stage": stage,
        "train_auc_pr": round(auc_pr_eval.evaluate(train_preds), 4),
        "val_auc_pr": m_val["auc_pr"],
        "train_val_gap": None,   # filled on the next line
        "val_auc_roc": m_val["auc_roc"],
        "train_accuracy": prediction_scores(train_preds)["accuracy"],
        "val_accuracy": m_val["accuracy"],
        "val_precision_churn": m_val["precision_churn"],
        "val_recall_churn": m_val["recall_churn"],
        "val_f1_churn": m_val["f1_churn"],
    }
    row["train_val_gap"] = round(row["train_auc_pr"] - row["val_auc_pr"], 4)
    results.append(row)
    return row


def compare_table(name):
    """Baseline and tuned rows of one model, for the before/after sections."""
    return pd.DataFrame([r for r in results if r["model"] == name])


def show_confusion(preds, title):
    """Print the churn-class metrics and show the confusion matrix (rows = prediction)."""
    m = churn_metrics(preds)
    print(title, "-", {k: v for k, v in m.items() if k != "confusion"})
    cm = m["confusion"]
    display(spark.createDataFrame(
        [(0, cm["TN"], cm["FP"]), (1, cm["FN"], cm["TP"])],
        ["predicted", "actually_stayed", "actually_churned"],
    ))


def plot_train_val(ax, groups, train_scores, val_scores, title, ref=None, ref_label=None):
    """Two bars per group (training dark, validation green), scores on top.

    `ref` optionally draws a dashed reference line — e.g. the random-ranking
    AUC-PR floor (the churn rate) from Step 2 — instead of a hard-coded one."""
    x = range(len(groups))
    width = 0.4
    for offset, values, color in [(-width / 2, train_scores, COLOR_DARK),
                                  (width / 2, val_scores, COLOR_MAIN)]:
        bars = ax.bar([i + offset for i in x], values, width, color=color)
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + width / 2, bar.get_height() + 0.01, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=9)
    handles = [
        Patch(color=COLOR_DARK, label="training"),
        Patch(color=COLOR_MAIN, label="validation"),
    ]
    if ref is not None:
        ax.axhline(ref, color=COLOR_GREY, linestyle="--", linewidth=1)
        handles.append(Line2D([0], [0], color=COLOR_GREY, linestyle="--",
                              label=f"{ref_label} ({ref:.3f})" if ref_label else f"{ref:.3f}"))
    ax.set_xticks(list(x))
    ax.set_xticklabels(groups)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(title)
    ax.legend(handles=handles, loc="lower right", fontsize=9)

## Step 3 — Baseline: five models with default settings

**What we do:** train one version of each model with Spark's default settings and our fixed preparation steps, all fitted on the 7,000 training rows — and compare all **five** in one table, on training and validation **AUC-PR** (Step 2's headline metric), with the churn-class precision / recall / F1 alongside.

| Model | What it is, in one sentence |
|---|---|
| **Logistic regression** | One weighted sum of the inputs, pushed through a curve to give a churn probability; draws a single straight decision boundary. |
| **Random Forest** | Many decision trees, each fitted on a random sample of rows and inputs; their votes are averaged. |
| **Gradient-Boosted Trees (GBT)** | Decision trees fitted one after another, each new tree correcting the mistakes of the trees so far. |
| **Linear SVM** | A single straight boundary like logistic regression, but fitted by maximising the *margin* — the distance between the boundary and the closest customers — instead of probabilities. |
| **Neural network (MLP)** | A small feed-forward network: one hidden layer of learned mixes of the inputs, which can combine inputs (age *and* products together) instead of weighing them separately — our first non-linear model that is not a tree. |


**Reading the scores:** every AUC-PR number is read against the **random-ranking floor ≈ 0.204** — the churn rate (the dashed reference line in the chart) — because a model that ranked customers at random would score exactly that. The churn-class recall then answers, at threshold 0.5, the question the curve summarises: of the customers who actually left, how many did the model flag? One model needs a spelling out: the SVM produces no churn *probability* at all, so no precision–recall curve — and no AUC-PR — exists for it; it appears with "—" in the table and is judged on the prediction-column metrics only.




In [0]:
# The network's `layers` setting has no default: one hidden layer the size of the
# prepared input vector and 2 outputs (stay / leave) — the smallest reasonable choice.
prep_only = Pipeline(
    stages=build_preprocessing_stages(CATEGORICAL_FEATURES, NUMERIC_FEATURES)
).fit(train)
n_inputs = len(prep_only.transform(train.limit(1)).select("features").first()[0])
print(f"Input vector after preparation: {n_inputs} features")

# The five baseline models: Spark's default settings, apart from labelCol (required
# by our pipeline), seed = SEED where the algorithm is random, and the network's layers.
baseline_clfs = [
    ("Logistic Regression", LogisticRegression(labelCol=LABEL_COL)),
    ("Random Forest", RandomForestClassifier(labelCol=LABEL_COL, seed=SEED)),
    ("Gradient-Boosted Trees", GBTClassifier(labelCol=LABEL_COL, seed=SEED)),
    ("Linear SVM", LinearSVC(labelCol=LABEL_COL)),                     # deterministic: no seed
    ("Neural Net (MLP)", MultilayerPerceptronClassifier(
        labelCol=LABEL_COL, layers=[n_inputs, n_inputs, 2], seed=SEED,
    )),
]

# Re-running this cell must not duplicate baseline rows in `results`.
results[:] = [r for r in results if r["stage"] != "baseline"]

baseline_models = {}
for name, clf in baseline_clfs:
    baseline_models[name] = make_pipeline(clf).fit(train)
    train_preds = baseline_models[name].transform(train)
    val_preds = baseline_models[name].transform(val)
    if name == "Linear SVM":
        # No probability column, so AUC-PR cannot be computed: the row keeps the
        # same table columns (AUC-PR empty) and is scored from the prediction column.
        m_train, m_val = prediction_scores(train_preds), prediction_scores(val_preds)
        results.append({
            "model": name, "stage": "baseline",
            "train_auc_pr": None, "val_auc_pr": None,
            "train_val_gap": None, "val_auc_roc": None,
            "train_accuracy": m_train["accuracy"], "val_accuracy": m_val["accuracy"],
            "val_precision_churn": m_val["precision_churn"],
            "val_recall_churn": m_val["recall_churn"],
            "val_f1_churn": m_val["f1_churn"],
        })
        print(f"{name}: trained; scored from predictions (no probability column for AUC-PR).")
    else:
        score_row(name, train_preds, val_preds, "baseline")
        print(f"{name}: trained and scored.")

# One table, all five models: validation AUC-PR is the headline (Step 2), with the
# churn-class metrics alongside and the train-validation AUC-PR gap as the first
# overfitting signal. The SVM has no probability column, so its AUC-PR is null.
baseline_table = pd.DataFrame([r for r in results if r["stage"] == "baseline"]).copy()
display(spark.createDataFrame(
    baseline_table[[
        "model", "train_auc_pr", "val_auc_pr", "train_val_gap",
        "val_precision_churn", "val_recall_churn", "val_f1_churn",
    ]]
))

In [0]:
# Training vs validation AUC-PR for the four baselines that produce a churn probability
# (the Linear SVM cannot be ranked — no probability column, Step 3), with the
# random-ranking floor from Step 2 — AUC-PR ≈ the churn rate — as the dashed reference line.
chart_table = baseline_table[baseline_table["val_auc_pr"].notna()]
fig, ax = plt.subplots(figsize=(9, 5))
plot_train_val(
    ax,
    list(chart_table["model"]),
    list(chart_table["train_auc_pr"]),
    list(chart_table["val_auc_pr"]),
    "Baseline models (default settings): training vs validation AUC-PR",
    ref=CHURN_RATE, ref_label="Random ranking (churn rate)",
)
plt.tight_layout()
plt.show()

**Result:** all four probability models more than triple the 0.204 random-ranking floor — and the two families carried into tuning stand out in different ways.

| Model | Training AUC-PR | Validation AUC-PR | Gap | Precision (churn) | Recall (churn) | F1 (churn) |
|---|---|---|---|---|---|---|
| Random ranking | — | 0.204 | — | — | — | — |
| Linear SVM | — | — | — | 1.000 | 0.016 | 0.032 |
| Logistic regression | 0.631 | 0.604 | 0.027 | 0.684 | 0.390 | 0.497 |
| Random Forest | 0.686 | 0.638 | 0.048 | 0.803 | 0.374 | 0.510 |
| Gradient-Boosted Trees | 0.771 | 0.656 | 0.115 | 0.746 | 0.462 | 0.571 |
| **Neural Net (MLP)** | 0.754 | **0.674** | 0.080 | 0.706 | **0.489** | **0.578** |

(The reference row is the random-ranking floor from Step 2: a model that ranks customers at random scores AUC-PR ≈ the churn rate. The SVM produces no probability column, so no precision–recall curve — and no AUC-PR — exists for it; its prediction-column metrics are shown for completeness.)

- **The neural network is the best baseline on the headline metric (validation AUC-PR 0.674)** — and it also has the best churn recall (48.9%, 149 of the 305 leavers) and the best F1 (0.578). Under accuracy it ranked second, behind GBT (0.855 vs 0.859) — exactly the Step 2 warning: accuracy rewards the model that flags fewer customers, AUC-PR rewards the better *ranking* of leavers.
- **GBT is second (0.656)** and keeps the best churn precision of the tree models (0.746) — but it already shows the largest train–validation gap of the five (0.115), the first memorisation signal.
- **The gaps are modest (0.027–0.115):** with default settings nothing is badly memorising the training rows yet.


## Step 4 — Hyperparameter tunning: Gradient-Boosted Trees

**How GBT works:** trees are fitted one after another, each concentrating on the rows the trees so far get wrong; the final score sums all trees, each shrunk by the learning rate `stepSize`. Good at fine patterns — and, with too many or too deep trees, the quickest of the five to start memorising.


**What we tune:** `maxIter` (20, 50, 100) × `maxDepth` (3, 5) × `stepSize` (0.05, 0.1) — **12 candidates**. Each is judged by **3-fold cross-validation inside the training split, by AUC-PR**, with the preparation steps fitted inside each fold's training rows only (nothing leaks); the winner is refitted on all 7,000 training rows and scored once on validation.

In [0]:
gbt = GBTClassifier(labelCol=LABEL_COL, seed=SEED)
gbt_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxIter, [20, 50, 100])
    .addGrid(gbt.maxDepth, [3, 5])
    .addGrid(gbt.stepSize, [0.05, 0.1])
    .build()
)

gbt_cv = CrossValidator(
    estimator=make_pipeline(gbt),   # the SAME gbt instance the grid refers to
    estimatorParamMaps=gbt_grid,
    evaluator=auc_pr_eval,         # selection by AUC-PR — Step 2's headline metric
    numFolds=3,
    seed=SEED,
)
gbt_cv_model = gbt_cv.fit(train)

# Grid-search result: every candidate's average AUC-PR across the three folds, best first.
gbt_cv_table = pd.DataFrame(
    [
        {"maxIter": params[gbt.maxIter], "maxDepth": params[gbt.maxDepth],
         "stepSize": params[gbt.stepSize], "cv_auc_pr": round(score, 4)}
        for params, score in zip(gbt_grid, gbt_cv_model.avgMetrics)
    ]
).sort_values("cv_auc_pr", ascending=False)
display(spark.createDataFrame(gbt_cv_table))

gbt_best = gbt_cv_model.bestModel.stages[-1]
# stepSize is captured too, so the refit below trains exactly the CV winner —
# not its shape with a default learning rate.
GBT_BEST = {
    "maxIter": gbt_best.getMaxIter(),
    "maxDepth": gbt_best.getMaxDepth(),
    "stepSize": gbt_best.getStepSize(),
}
print(
    f"Selected parameters: maxIter={GBT_BEST['maxIter']}, "
    f"maxDepth={GBT_BEST['maxDepth']}, stepSize={GBT_BEST['stepSize']}"
)

In [0]:
# Refit the selected settings on ALL 7,000 training rows, then score train and validation.
gbt_tuned = make_pipeline(
    GBTClassifier(labelCol=LABEL_COL, seed=SEED, **GBT_BEST)
).fit(train)
# Re-running this cell replaces its row instead of adding a duplicate.
results[:] = [r for r in results if not (r["model"] == "Gradient-Boosted Trees" and r["stage"] == "tuned")]
score_row("Gradient-Boosted Trees", gbt_tuned.transform(train), gbt_tuned.transform(val), "tuned")

gbt_compare = compare_table("Gradient-Boosted Trees").copy()
display(spark.createDataFrame(
    gbt_compare[["stage", "train_auc_pr", "val_auc_pr", "train_val_gap", "val_recall_churn", "val_f1_churn"]]
))

fig, ax = plt.subplots(figsize=(8, 5))
plot_train_val(
    ax, ["Baseline", "Tuned"],
    list(gbt_compare["train_auc_pr"]), list(gbt_compare["val_auc_pr"]),
    "Gradient-Boosted Trees: AUC-PR before vs after tuning",
    ref=CHURN_RATE, ref_label="Random ranking (churn rate)",
)
plt.tight_layout()
plt.show()

**Result:** tuning helps a little. The grid picks `maxIter=100, maxDepth=3, stepSize=0.1` (best CV AUC-PR: 0.707). Refit on all 7,000 training rows, it scores validation AUC-PR **0.661**, slightly above the baseline's **0.656**. Churn recall goes 0.462 → 0.472 (141 → 144 of 305), false alarms drop (48 → 44) and the train–validation gap shrinks 0.115 → 0.095.


## Step 5 — Hyperparameter tunning: Neural network (MLP)

**How the network works:** The 16 prepared inputs feed a hidden layer of neurons, each a learned mix of the inputs pushed through a non-linear function. Unlike a tree, a neuron can combine inputs multiplicatively (age *and* products together), so the network captures patterns like the U-shaped product effect in a different way than the tree models — which is why its mistakes differ from theirs.


**What we tune, and why:**
- **A — the baseline itself:** `[16, 16, 2]` — one hidden layer the size of the input. Included deliberately: the grid can only *match or beat* the architecture Step 3 already showed to be the best — "keep the baseline" must be an allowed answer, otherwise the search is forced to change something even when change is not an improvement.
- **B — one wider hidden layer:** `[16, 32, 2]` — twice the baseline's width, still a single layer.
- **C — two hidden layers:** `[16, 16, 8, 2]` — a second, narrower layer to combine what the first layer learned.

All three start from the prepared input vector (its size is derived from the feature vector) and end in **2 output units** (stay / leave), as a binary classifier requires; `maxIter` and the seed stay at their defaults. Three candidates, judged with the same **3-fold cross-validation inside the training split, by AUC-PR**, with the preparation steps fitted inside each fold's training rows only. The winner is refitted on all 7,000 training rows and scored once on the validation split.

In [0]:
mlp = MultilayerPerceptronClassifier(labelCol=LABEL_COL, seed=SEED)

# Three simple hidden-layer configurations: the input size comes from the prepared
# feature vector (n_inputs, from Step 3) and the output layer has 2 units (stay / leave).
# A is the baseline architecture itself — the grid can only match or beat Step 3's choice.
mlp_grid = (
    ParamGridBuilder()
    .addGrid(mlp.layers, [
        [n_inputs, n_inputs, 2],
        [n_inputs, 2 * n_inputs, 2],
        [n_inputs, n_inputs, max(1, n_inputs // 2), 2]
    ])
    .addGrid(mlp.maxIter, [100, 200])
    .build()
)

mlp_cv = CrossValidator(
    estimator=make_pipeline(mlp),   # the SAME mlp instance the grid refers to
    estimatorParamMaps=mlp_grid,
    evaluator=auc_pr_eval,         # selection by AUC-PR — Step 2's headline metric
    numFolds=3,
    seed=SEED,
)
mlp_cv_model = mlp_cv.fit(train)

# Grid-search result: each architecture's average AUC-PR across the three folds.
mlp_cv_table = pd.DataFrame(
    [
        {"layers": str(params[mlp.layers]), "cv_auc_pr": round(score, 4)}
        for params, score in zip(mlp_grid, mlp_cv_model.avgMetrics)
    ]
).sort_values("cv_auc_pr", ascending=False)
display(spark.createDataFrame(mlp_cv_table))

mlp_best = mlp_cv_model.bestModel.stages[-1]
MLP_BEST = {"layers": mlp_best.getLayers()}
print("Selected layers:", MLP_BEST["layers"])

In [0]:
# Refit the selected architecture on ALL 7,000 training rows; compare with its baseline.
mlp_tuned = make_pipeline(
    MultilayerPerceptronClassifier(labelCol=LABEL_COL, seed=SEED, **MLP_BEST)
).fit(train)
# Re-running this cell replaces its row instead of adding a duplicate.
results[:] = [r for r in results if not (r["model"] == "Neural Net (MLP)" and r["stage"] == "tuned")]
score_row("Neural Net (MLP)", mlp_tuned.transform(train), mlp_tuned.transform(val), "tuned")

mlp_compare = compare_table("Neural Net (MLP)").copy()
display(spark.createDataFrame(
    mlp_compare[["stage", "train_auc_pr", "val_auc_pr", "train_val_gap", "val_recall_churn", "val_f1_churn"]]
))

fig, ax = plt.subplots(figsize=(8, 5))
plot_train_val(
    ax, ["Baseline", "Tuned"],
    list(mlp_compare["train_auc_pr"]), list(mlp_compare["val_auc_pr"]),
    "Neural network (MLP): AUC-PR before vs after tuning",
    ref=CHURN_RATE, ref_label="Random ranking (churn rate)",
)
plt.tight_layout()
plt.show()

**Result:** with the baseline allowed as a candidate, the grid search keeps it. Cross-validated AUC-PR selects **the baseline architecture** `[16, 16, 2]` (CV AUC-PR 0.652) over the wider layer `[16, 32, 2]` (0.643) and the two-hidden-layer network `[16, 16, 8, 2]` (0.636). The tuned model is therefore the baseline model: validation AUC-PR stays **0.674**, churn recall **0.489** (149 of the 305 leavers), F1 **0.578**, train–validation gap **0.080** — nothing was traded away.

| | Training AUC-PR | Validation AUC-PR | Gap | Recall (churn) |
|---|---|---|---|---|
| Baseline `[16, 16, 2]` = **tuned** | 0.754 | **0.674** | 0.080 | 0.489 (149/305) |



## Step 6 — Model comparison

**What we do:** put the tuned Gradient-Boosted Trees and the tuned neural network side by side — training and validation **AUC-PR**, the train–validation AUC-PR gap, and the churn-class recall and F1 on validation — and answer the two questions that matter for the project discussion: **did tuning improve validation AUC-PR over each model's own baseline, and did churn recall improve or decline?**


In [0]:
tuned_table = pd.DataFrame([
    r for r in results
    if r["stage"] == "tuned" and r["model"] in ("Neural Net (MLP)", "Gradient-Boosted Trees")
]).copy()
display(spark.createDataFrame(
    tuned_table[[
        "model", "train_auc_pr", "val_auc_pr", "train_val_gap",
        "val_precision_churn", "val_recall_churn", "val_f1_churn",
    ]]
))

**The preferred model depends on the bank’s objective.** The MLP achieves slightly higher validation AUC-PR and recall, while GBT achieves higher precision and F1.

| Model | Training AUC-PR | Validation AUC-PR | Gap | Precision (churn) | Recall (churn) | F1 (churn) |
|---|---|---|---|---|---|---|
| **Neural Network (MLP)** | 0.754 | **0.674** | 0.080 | 0.706 | **0.489** | 0.578 |
| **Gradient-Boosted Trees** | 0.756 | 0.661 | 0.095 | **0.766** | 0.472 | **0.584** |

Both models substantially exceed the approximate random-ranking reference of **0.204**, corresponding to the validation churn proportion. The train–validation gaps suggest some overfitting, slightly more pronounced for GBT.

Tuning increased GBT’s validation AUC-PR from **0.656 to 0.661**. The MLP search retained the baseline architecture, with validation AUC-PR remaining at **0.674**.

### Which model best supports our objective?

At the current thresholds, out of **305 actual churners**:

- **MLP** identifies 149 churners and generates 62 false positives. It may be preferable when the priority is to reach more customers at risk and additional contacts are affordable.
- **GBT** identifies 144 churners and generates 44 false positives. It may be preferable when retention resources are limited or unnecessary offers are costly.

The MLP identifies **5 additional churners at the cost of 18 additional false positives**. Choosing between them therefore requires balancing the potential value of reaching those customers against the cost of the additional contacts.

**If validation AUC-PR is our selection criterion, the MLP is the leading candidate, although its advantage is small.** For operational use, the choice should reflect the bank’s retention objective and available resources.



**Model stability (checked separately):** Each tuned model was retrained with five seeds (0, 1, 2, 3 and 42) on the same data, split and preparation.

GBT is deterministic — validation AUC-PR 0.654 on every seed (`subsamplingRate=1.0`, so the seed has nothing to randomise). The MLP varies a little: AUC-PR 0.675 ± 0.006 (worst 0.668), churn recall 0.496 ± 0.012. Even its worst seed is above the GBT, so the Step 6 comparison holds in every run — and the seed-42 runs reproduce those numbers exactly.

## Step 7 — Stacking 

**Motivation:** After hyperparameter tuning, the two best-performing models were combined using a stacking approach. The motivation behind this strategy is that different models may capture complementary patterns in the data. By combining their predictions, it is possible to leverage their individual strengths and potentially improve overall performance. A Logistic Regression model was used as the final estimator to learn how to best combine the outputs of the base models.



**Honest expectation:** stacking only helps if the two models make *different* mistakes. We do not assume it will win — the comparison below shows whether it does, and if it does not, we report that plainly.

In [0]:
# Stacking is ON HOLD: with RUN_STACKING = False, this cell only sets the switch and
# the section's dependencies — the ensemble work below is skipped.
RUN_STACKING = True

if RUN_STACKING:
    from pyspark.ml.feature import VectorAssembler

    N_STACK_FOLDS = 3


    def add_stack_fold(df, n_folds=N_STACK_FOLDS, seed=SEED):
        """Assign every training row to one of n folds (0..n-1), stratified by Exited.

        The same trick as src/ml.py's add_split_column: order rows by a hash of
        (CustomerId, a different seed) inside each label group, then cut the order into
        equal parts. Deterministic, and every fold keeps the same churn rate.
        """
        order = F.xxhash64(F.col(KEY_COL).cast("string"), F.lit(seed + 999))
        position = F.percent_rank().over(Window.partitionBy(LABEL_COL).orderBy(order))
        # percent_rank reaches exactly 1.0 for the last row of each group, which would
        # give floor(1.0 * 3) = 3 - a fold that does not exist. `least` caps it at 2,
        # so every single row lands in a real fold.
        fold_number = F.least(F.floor(position * n_folds).cast("int"), F.lit(n_folds - 1))
        return df.withColumn("stack_fold", fold_number)


    def oof_churn_probability(classifier, prob_col):
        """Out-of-fold churn probabilities for one base model inside the training split.

        For each of the 3 folds: fit preparation + model on the OTHER two folds (so the
        preparation is also fitted fold by fold), then predict the held-out fold.
        Every training row ends up with a probability from a model that never saw it.
        """
        folded = add_stack_fold(train).cache()
        parts = []
        for fold in range(N_STACK_FOLDS):
            taught = folded.filter(F.col("stack_fold") != fold)
            held_out = folded.filter(F.col("stack_fold") == fold)
            fold_model = make_pipeline(classifier).fit(taught)
            parts.append(
                fold_model.transform(held_out)
                .select(KEY_COL, LABEL_COL,
                        vector_to_array("probability")[1].cast("double").alias(prob_col))
            )
        out = parts[0]
        for part in parts[1:]:
            out = out.unionByName(part)
        return out


    # The two tuned base models, built fresh (they are fitted once per fold, so we
    # pass the settings, not fitted models): the GBT winner from Step 4 and the MLP
    # winner from Step 5 — a tree ensemble and a neural network, which the Step 6
    # comparison showed make different mistakes (precision vs recall).
    stack_gbt = GBTClassifier(labelCol=LABEL_COL, seed=SEED, **GBT_BEST)
    stack_mlp = MultilayerPerceptronClassifier(labelCol=LABEL_COL, seed=SEED, **MLP_BEST)

    oof_gbt = oof_churn_probability(stack_gbt, "p_gbt")
    oof_mlp = oof_churn_probability(stack_mlp, "p_mlp")

    meta_train = (
        oof_gbt.join(oof_mlp, [KEY_COL, LABEL_COL], "inner")
        .select(KEY_COL, LABEL_COL, "p_gbt", "p_mlp")
    )
    print(f"Out-of-fold training rows for the meta-model: {meta_train.count():,}")
    meta_train = VectorAssembler(
        inputCols=["p_gbt", "p_mlp"], outputCol="features"
    ).transform(meta_train).cache()

In [0]:
if RUN_STACKING:
    # 1) Meta-model: a plain logistic regression on the two out-of-fold probabilities.
    meta_model = LogisticRegression(labelCol=LABEL_COL).fit(meta_train)
    print(
        "Meta-model: p_churn = sigmoid("
        f"{meta_model.intercept:.3f} + {meta_model.coefficients[0]:.3f}*p_gbt"
        f" + {meta_model.coefficients[1]:.3f}*p_mlp)"
    )

    # 2) Refit both base models on the FULL training split; their probabilities on any
    #    new data are the meta-model's inputs.
    base_gbt_full = make_pipeline(
        GBTClassifier(labelCol=LABEL_COL, seed=SEED, **GBT_BEST)
    ).fit(train)
    base_mlp_full = make_pipeline(
        MultilayerPerceptronClassifier(labelCol=LABEL_COL, seed=SEED, **MLP_BEST)
    ).fit(train)


    def meta_features(data):
        """Churn probabilities of both base models for any customer set -> one vector."""
        gbt_p = (
            base_gbt_full.transform(data)
            .select(KEY_COL, LABEL_COL, vector_to_array("probability")[1].cast("double").alias("p_gbt"))
        )
        mlp_p = (
            base_mlp_full.transform(data)
            .select(KEY_COL, vector_to_array("probability")[1].cast("double").alias("p_mlp"))
        )
        joined = gbt_p.join(mlp_p, KEY_COL).select(KEY_COL, LABEL_COL, "p_gbt", "p_mlp")
        return VectorAssembler(inputCols=["p_gbt", "p_mlp"], outputCol="features").transform(joined)


    # 3) Score the ensemble. "Training" here means: the meta-model applied to the
    #    out-of-fold features of the training rows (see the Result cell for the caveat).
    # Re-running this cell replaces its row instead of adding a duplicate.
    results[:] = [r for r in results if r["model"] != "Stacking (GBT + MLP)"]
    stack_train_preds = meta_model.transform(meta_train)
    stack_val_preds = meta_model.transform(meta_features(val))
    score_row("Stacking (GBT + MLP)", stack_train_preds, stack_val_preds, "tuned")

    stack_compare = pd.DataFrame([
        r for r in results
        if r["stage"] == "tuned"
        and r["model"] in ("Neural Net (MLP)", "Gradient-Boosted Trees", "Stacking (GBT + MLP)")
    ])
    display(spark.createDataFrame(stack_compare[[
        "model", "train_auc_pr", "val_auc_pr", "train_val_gap",
        "val_precision_churn", "val_recall_churn", "val_f1_churn",
    ]]))
    show_confusion(stack_val_preds, "Stacking (GBT + MLP) on validation")
else:
    print(
        "Stacking is on hold (RUN_STACKING = False) — skipped.\n"
        "Set RUN_STACKING = True in the cell above and re-run this section to enable it."
    )

## Step 8 — The chosen model

**Decision:** the final model is the **MLP classifier** — the tuned network, which keeps the baseline architecture `[16, 16, 2]`.

It has the best validation AUC-PR of everything we tried (0.674, against 0.661 for the tuned GBT and 0.667 for the stacking ensemble), the best churn recall (149 of the 305 leavers), and it is stable across seeds (0.675 ± 0.006). Everything from here on — the operating threshold, the test set — uses this model.

## Step 9 — Choosing the operating threshold

**What we do:** the tuned network gives every customer a churn *probability*; AUC-PR judged the ranking, but the bank needs a contact list — a yes/no per customer. The **threshold** is the cut the probability is compared against (contact the customer if p_churn ≥ threshold), and it is a business decision, not a modelling one. We sweep 0.05–0.95 on **validation only** (the test set stays closed, Step 1's rule) and read off three choices.

**Best F1** — the threshold that balances churn precision and recall. **Best cost** — the threshold that minimises an illustrative expected cost: a wasted retention offer costs 1 unit, a missed leaver 5 (placeholder numbers — swap in the bank's real economics). **Budget view** — contacting the riskiest 10% of customers is itself a threshold: the 90th percentile of the scores.

In [0]:
# The operating threshold turns a probability into a decision: contact the customer
# if p_churn >= threshold. Chosen on VALIDATION only — the test set stays closed.
# Swap FINAL_MODEL to run the same analysis for any model (e.g. gbt_tuned).
FINAL_MODEL = mlp_tuned              # Step 6: the tuned network, the notebook's best model

# Illustrative economics for the cost-based choice — replace with the bank's real numbers.
COST_OFFER = 1.0                     # one wasted retention offer
COST_MISSED = 5.0                    # one leaver the bank never contacted

val_scores = FINAL_MODEL.transform(val).select(
    LABEL_COL,
    vector_to_array("probability")[1].cast("double").alias("p_churn"),
).cache()

# All 19 thresholds (0.05 .. 0.95) tested in ONE pass: each row becomes 19 (row,
# threshold) pairs, aggregated in a single shuffle.
thresholds = [round(0.05 * i, 2) for i in range(1, 20)]
sweep = (
    val_scores
    .withColumn("t", F.explode(F.array(*[F.lit(t) for t in thresholds])))
    .groupBy("t")
    .agg(
        F.sum(((F.col("p_churn") >= F.col("t")) & (F.col(LABEL_COL) == 1)).cast("int")).alias("tp"),
        F.sum(((F.col("p_churn") >= F.col("t")) & (F.col(LABEL_COL) == 0)).cast("int")).alias("fp"),
        F.sum(((F.col("p_churn") <  F.col("t")) & (F.col(LABEL_COL) == 1)).cast("int")).alias("fn"),
    )
    .select(
        "t", "tp", "fp", "fn",
        F.round(F.when(F.col("tp") + F.col("fp") > 0,
                       F.col("tp") / (F.col("tp") + F.col("fp"))).otherwise(0.0), 4
                ).alias("precision_churn"),
        F.round(F.when(F.col("tp") + F.col("fn") > 0,
                       F.col("tp") / (F.col("tp") + F.col("fn"))).otherwise(0.0), 4
                ).alias("recall_churn"),
        F.round(F.lit(COST_OFFER) * F.col("fp") + F.lit(COST_MISSED) * F.col("fn"), 1
                ).alias("expected_cost"),
    )
    .withColumn(
        "f1_churn",
        F.round(F.when(F.col("precision_churn") + F.col("recall_churn") > 0,
                       2 * F.col("precision_churn") * F.col("recall_churn")
                       / (F.col("precision_churn") + F.col("recall_churn"))).otherwise(0.0), 4),
    )
    .orderBy("t")
)
display(sweep)

# The three headline choices from the sweep.
best_f1 = sweep.orderBy(F.col("f1_churn").desc(), F.col("t")).first()
best_cost = sweep.orderBy(F.col("expected_cost"), F.col("t")).first()
top10_cut = float(val_scores.agg(F.percentile_approx("p_churn", 0.9, 10000)).first()[0])
THRESHOLD_BEST_F1 = float(best_f1["t"])
THRESHOLD_BEST_COST = float(best_cost["t"])
print(f"Best F1:   threshold {THRESHOLD_BEST_F1:.2f} — precision {best_f1['precision_churn']}, "
      f"recall {best_f1['recall_churn']}, F1 {best_f1['f1_churn']}")
print(f"Best cost: threshold {THRESHOLD_BEST_COST:.2f} — precision {best_cost['precision_churn']}, "
      f"recall {best_cost['recall_churn']}, expected cost {best_cost['expected_cost']:.0f} units "
      f"({COST_OFFER:.0f}/wasted offer, {COST_MISSED:.0f}/missed leaver)")
print(f"Budget:    contacting the riskiest 10% = threshold {top10_cut:.3f}")

# Precision / recall / F1 against the threshold, with the default (0.5) and the
# best-F1 cut marked.
rows = sweep.collect()
ts = [r["t"] for r in rows]
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ts, [r["precision_churn"] for r in rows], marker="o", ms=3, color=COLOR_DARK, label="Precision (churn)")
ax.plot(ts, [r["recall_churn"] for r in rows], marker="o", ms=3, color=COLOR_MAIN, label="Recall (churn)")
ax.plot(ts, [r["f1_churn"] for r in rows], marker="o", ms=3, color="0.3", ls="--", label="F1 (churn)")
ax.axvline(0.5, color="0.6", lw=1, ls=":", label="Spark default (0.5)")
ax.axvline(THRESHOLD_BEST_F1, color=COLOR_MAIN, lw=1.5, ls="--", label=f"Best F1 ({THRESHOLD_BEST_F1:.2f})")
ax.set_xlabel("Threshold on p_churn")
ax.set_ylabel("Validation score")
ax.set_title("The operating threshold: precision, recall and F1 vs threshold (validation)")
ax.legend()
plt.tight_layout()
plt.show()

**Result:** the default 0.5 is not the best cut for any of the three goals. Lowering it to **0.35** lifts churn F1 from 0.578 to **0.612**: the model flags 287 instead of 211 customers and catches 181 instead of 149 of the 305 leavers, at the price of 106 instead of 62 false alarms. If a missed leaver costs five times a wasted offer (placeholder economics), the cost-optimal cut is **0.20**: 233 of 305 leavers caught (recall 0.764) and an expected cost of 602 units instead of 842 at the default. And the budget view: contacting the riskiest 10% of customers means a threshold of **0.651**.

| Choice | Threshold | Precision (churn) | Recall (churn) | F1 (churn) | Expected cost (1:5) |
|---|---|---|---|---|---|
| Spark default | 0.50 | 0.706 | 0.489 | 0.578 | 842 |
| Best F1 | 0.35 | 0.631 | 0.593 | **0.612** | 726 |
| Best cost (1:5) | 0.20 | 0.491 | **0.764** | 0.597 | **602** |

**Decision: we choose 0.35** — the best-F1 cut. With no retention costs or contact budget defined, the defensible criterion is the balanced one: the threshold that maximises churn-class F1 on validation (0.612, against 0.578 at the default). The cost-based and budget cuts stay in the table for the day the bank supplies real economics — re-deciding then means re-reading this table, not re-training the model.